# "Flux Already Knows" → Modular Diffusers — E2E (training-free subject-driven generation)

Put a reference subject into 2–3 new scenes with **no training and no extra weights** (LatentUnfold, arXiv:2504.11478).
Publish PRIVATE `remyxai/flux-subject-flux-modular` → load via `trust_remote_code` → generate → **quantitative validation**: subject fidelity (**CLIP-I** vs the reference, and **DreamSim** if available) **AND** prompt adherence (**CLIP** to the scene text) — both must hold. Then a **strength sweep** to check the StyleAligned-style over-collapse (subject fidelity up, prompt ignored), and a **head-to-head vs PuLID** on a subject PuLID can't do (a non-face).
Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.
**VRAM:** the cascade path materializes the full attention matrix (~4.2 GB/layer at 3x3 with 512px
tiles, on top of ~24 GB of weights). On a 40 GB A100, drop to `height=width=384` or
`grid_shape=(2,2)`.

## 1 · Install + GPU + auth


In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf


In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16


## 2 · Publish PRIVATE (upload block.py first)


In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"FluxSubjectBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.FluxSubjectBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"FluxSubjectBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/flux-subject-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))


## 3 · Load + the reference subject


In [ ]:
from diffusers import ModularPipeline
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/flux-subject-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "FluxSubjectBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect FluxSubjectBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

IMG_URL = "https://raw.githubusercontent.com/bytedance/LatentUnfold/main/assets/clock1.jpg"  #@param {type:"string"}
try:
    subj = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up=files.upload(); subj = Image.open(list(up.keys())[0]).convert("RGB")
subj.save("subject.png")
print("reference subject:"); display(subj.resize((320,320)))


## 4 · The subject in 2–3 new scenes (full settings)


In [ ]:
import torch
SUBJECT = "bright yellow retro alarm clock"  #@param {type:"string"}
SCENES = [
    "In a Bauhaus style room, this item is placed on a shiny glass table, with a vase of flowers next to it.",
    "On a beach at sunset, half buried in golden sand, waves lapping in the background.",
    "On a wooden desk in a cozy study, next to a stack of books and a cup of coffee.",
]
outs = [("reference", subj)]
for i, sc in enumerate(SCENES):
    g = torch.Generator(DEV).manual_seed(0)
    im = pipe(subject_image="subject.png", prompt=sc, subject_prompt=SUBJECT,
              height=512, width=512, grid_shape=(3,3), num_inference_steps=28,
              guidance_scale=7.0, subject_strength=0.05, cascade=(2,3), injection_steps=14,
              generator=g).images[0]
    im.save(f"scene_{i}.png"); outs.append((f"scene {i+1}", im)); print("  ✓", sc[:60])

S, GAP = 384, 10
W = len(outs)*S + (len(outs)+1)*GAP
row = Image.new("RGB", (W, S+40), "white"); d = ImageDraw.Draw(row)
try: Ft = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 26)
except Exception: Ft = ImageFont.load_default()
for i, (name, im) in enumerate(outs):
    x = GAP + i*(S+GAP); row.paste(im.resize((S,S)), (x,0)); d.text((x+8, S+6), name, fill="black", font=Ft)
row.save("scenes.png"); print("reference | scenes:"); display(row.resize((min(W,1400), int((S+40)*min(W,1400)/W))))


## 5 · Quantitative — subject fidelity AND prompt adherence

The brief's two-sided gate. **CLIP-I** (image↔image, `openai/clip-vit-base-patch32` vision tower) against the reference
subject measures *identity*; **CLIP-T** against the scene text measures *prompt adherence*. A **stock FLUX** baseline
(same scenes, no reference) shows the fidelity the subject actually buys: ΔCLIP-I should be clearly positive while
CLIP-T stays comparable. DreamSim is added if the package installs.


In [ ]:
import numpy as np, torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
def _emb(img):
    b = proc(images=[img], return_tensors="pt", padding=True).to(DEV)
    with torch.no_grad(): return clip.get_image_features(**b)[0]
def sim(img, text):
    b = proc(text=[text], images=[img], return_tensors="pt", padding=True).to(DEV)
    with torch.no_grad(): o = clip(**b)
    ie = o.image_embeds/o.image_embeds.norm(dim=-1,keepdim=True); te = o.text_embeds/o.text_embeds.norm(dim=-1,keepdim=True)
    return float((ie@te.T)[0,0])
ref_e = _emb(subj)
def clip_i(img):
    e = _emb(img); return float(e @ ref_e / (e.norm()*ref_e.norm()))

dreamsim = None
try:
    import subprocess; subprocess.run(["pip","install","-q","dreamsim"], check=True)
    from dreamsim import dreamsim as _ds
    dreamsim = _ds(pretrained=True, device=DEV)
except Exception as e:
    print("(dreamsim unavailable, skipping — CLIP-I carries the fidelity signal)", e)

def d_sim(a, b):
    if dreamsim is None: return None
    with torch.no_grad(): return float(1.0 - dreamsim(a.resize((224,224)), b.resize((224,224))))

# stock FLUX baseline: same scenes, no reference subject, no mosaic
base = []
for i, sc in enumerate(SCENES):
    g = torch.Generator(DEV).manual_seed(0)
    base.append(pipe(subject_image="subject.png", prompt=f"{SUBJECT}. {sc}",
                     grid_shape=(1,1), height=512, width=512, num_inference_steps=28,
                     guidance_scale=3.5, generator=g).images[0])

print("scene | CLIP-I (subj) | ΔCLIP-I vs stock | CLIP-T | DreamSim (subj)")
ok = True
for i, sc in enumerate(SCENES):
    im = Image.open(f"scene_{i}.png")
    ci, bi, ct = clip_i(im), clip_i(base[i]), sim(im, sc)
    ds = d_sim(im, subj)
    print(f"  {i+1}:  CLIP-I={ci:.3f}  Δ={ci-bi:+.3f}  CLIP-T={ct:.3f}" + (f"  DreamSim={ds:.3f}" if ds is not None else ""))
    ok = ok and ci > bi                                 # the subject must actually transfer
print("\nsubject fidelity beats stock FLUX in every scene:", "PASS" if ok else "REVIEW")


## 6 · Spike — the over-collapse sweep (`subject_strength`)

The StyleAligned lesson: crank the sharing knob and subject fidelity climbs while the prompt is ignored. Cascade
attention is exactly such a knob. Sweep it and report both metrics — expect a plateau then a collapse; the README's
default (0.05) should sit on the plateau. Also shows `subject_strength=0.0`, the pure-mosaic no-op.


In [ ]:
import torch
from PIL import Image
SC = SCENES[0]
print("subject_strength | CLIP-I (subj) | CLIP-T (scene)")
for s in [0.0, 0.02, 0.05, 0.1, 0.2]:
    g = torch.Generator(DEV).manual_seed(0)
    im = pipe(subject_image="subject.png", prompt=SC, subject_prompt=SUBJECT, subject_strength=s,
              height=512, width=512, grid_shape=(3,3), num_inference_steps=28,
              guidance_scale=7.0, generator=g).images[0]
    im.save(f"sweep_{s}.png")
    print(f"  {s:>13}:  CLIP-I={clip_i(im):.3f}  CLIP-T={sim(im, SC):.3f}")
print("\nexpected: CLIP-I rises off 0.0 then saturates; CLIP-T roughly flat, dropping only at the top end.")
print("If CLIP-T falls off a cliff before CLIP-I saturates, lower the default subject_strength.")


## 7 · Head-to-head vs PuLID (a subject PuLID can't do)

PuLID needs trained **face** weights — this block is arbitrary subjects, weight-free. Same subject, same scene.
PuLID is given a fair text-only shot at a non-face subject; it should lose on CLIP-I here, which is the gap this
pipeline fills.


In [ ]:
import gc, torch
from PIL import Image
del pipe; gc.collect(); torch.cuda.empty_cache()

from diffusers import ModularPipeline
pl = ModularPipeline.from_pretrained("remyxai/pulid-flux-modular", trust_remote_code=True)
assert type(pl.blocks).__name__ == "PuLIDBlock", type(pl.blocks).__name__
pl.load_components(dtype=DT); pl.to(DEV)

SC = SCENES[0]; g = torch.Generator(DEV).manual_seed(0)
try:
    pu = pl(image="subject.png", prompt=f"{SUBJECT}. {SC}", height=512, width=512,
            num_inference_steps=28, generator=g).images[0]
    pu.save("pulid.png")
    ours = Image.open("scene_0.png")
    print(f"PuLID CLIP-I={clip_i(pu):.3f}  vs  this block CLIP-I={clip_i(ours):.3f}  ->",
          "WE WIN (expected off-face)" if clip_i(ours) > clip_i(pu) else "REVIEW")
except Exception as e:
    print("PuLID could not run on this subject (expected for a non-face):", type(e).__name__, str(e)[:160])
    print("That failure IS the gap this pipeline fills — record it in the PR.")


## Verdict
PASS = `loaded block: FluxSubjectBlock` + **ΔCLIP-I > 0 in every scene** (the subject transfers) + **CLIP-T comparable to stock** (the prompt still holds) + the strength sweep shows a plateau rather than a cliff + PuLID loses (or can't run) on a non-face subject. On pass: human confirms, flip the repo public, add the Colab badge + umbrella collection.
